# Extracting Spotify Audio Features via the API

**Fix notes (updated version):**
- **Credentials**: the original hardcoded `client_id`/`client_secret` are dead. This version reads them from environment variables (`SPOTIFY_CLIENT_ID`, `SPOTIFY_CLIENT_SECRET`), with a fallback cell to paste them directly if you'd rather not set env vars. Get your own free credentials at https://developer.spotify.com/dashboard.
- **Auth pattern modernized**: replaced the manual `get_access_token()` / token-caching pattern (fragile on newer spotipy) with `spotipy.Spotify(client_credentials_manager=...)`, which spotipy handles internally (including token refresh). Set up once, reused for the whole notebook — the original's duplicate re-authentication cells are no longer needed.
- **`tqdm_notebook` → `tqdm.notebook.tqdm`**: `tqdm_notebook` is deprecated and will be removed in tqdm 5.0.
- **Full dataset instead of `range(0, 1000)`**: the original URI-fetching loop only processed the first 1,000 rows. Replaced with a `SAMPLE_SIZE` toggle at the top — set it to a small number (e.g. `50`) to smoke-test your credentials quickly, or `None` to run the full dataset. Everything downstream (features, year, genre) naturally inherits whichever subset makes it through this first step, since each stage filters further.
- **Checkpointing**: the URI-fetching loop (the slowest, most failure-prone step — one live API call per song across potentially ~14,000 rows) now saves progress every 500 songs, so a crash or rate-limit doesn't cost you the whole run.
- **Consistent paths**: all inputs/outputs now live in `data/`, matching notebooks 1 and 2 (the original mixed `./data/...` and `../data/...`, which only worked if you ran cells from two different working directories).
- **"Get Year" step now uses the exact URI already matched** in the "Get Spotify URI" step (`sp.track(uri)`) instead of re-searching Spotify by title/artist a second time. This is faster (no second API call per song wasted on fuzzy search) and more accurate (no risk of a second search landing on a different track than the one you already matched).

- **Audio Features endpoint blocked for new apps**: Spotify restricted `sp.audio_features()` for any app created on/after Nov 27, 2024. This version tries the live API first, then falls back to `data/19000-spotify-songs/song_data.csv` (the Kaggle audio-features file) when the API call fails — see the "Get Spotify Audio Features" section for details and coverage caveats.
- **Premium-account requirement**: as of Feb 2026, new Development Mode Spotify apps require the app owner's account to have an active Premium subscription just to make basic calls like search — unrelated to the audio-features issue above, but worth knowing if your auth test cell fails with a 403 mentioning "premium subscription required".

In [24]:
import pandas as pd
import numpy as np
import os

## Config

Set `SAMPLE_SIZE` to a small integer (e.g. `50`) the first time you run this, to confirm your credentials and the pipeline work before committing to a full run across the whole dataset.

In [25]:
DATA_DIR = 'data'
SAMPLE_SIZE = 50  # set to None to run on the full dataset once you've confirmed everything works

## Spotify API Credentials

Get your own free credentials at https://developer.spotify.com/dashboard (create an app, copy its Client ID and Client Secret — the hardcoded ones from the original notebook are dead).

**Preferred**: set environment variables before launching Jupyter/VS Code:
- Windows PowerShell: `$env:SPOTIFY_CLIENT_ID="your_id"` and `$env:SPOTIFY_CLIENT_SECRET="your_secret"`
- macOS/Linux: `export SPOTIFY_CLIENT_ID=your_id` and `export SPOTIFY_CLIENT_SECRET=your_secret`

**Fallback**: if you'd rather not deal with environment variables, paste your credentials directly into the `CLIENT_ID_FALLBACK` / `CLIENT_SECRET_FALLBACK` variables below. Just don't commit this notebook to a public repo with real credentials pasted in.

In [26]:
CLIENT_ID_FALLBACK = ""      # paste your Client ID here if not using env vars
CLIENT_SECRET_FALLBACK = ""  # paste your Client Secret here if not using env vars

CLIENT_ID = os.environ.get("SPOTIFY_CLIENT_ID") or CLIENT_ID_FALLBACK
CLIENT_SECRET = os.environ.get("SPOTIFY_CLIENT_SECRET") or CLIENT_SECRET_FALLBACK

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "Spotify credentials not found. Either set the SPOTIFY_CLIENT_ID / "
        "SPOTIFY_CLIENT_SECRET environment variables, or paste your credentials "
        "into CLIENT_ID_FALLBACK / CLIENT_SECRET_FALLBACK above."
    )

ValueError: Spotify credentials not found. Either set the SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET environment variables, or paste your credentials into CLIENT_ID_FALLBACK / CLIENT_SECRET_FALLBACK above.

## Imports and Spotify Client Setup

One authenticated client (`sp`), reused for every API call in this notebook — no need to re-authenticate per section like the original did.

In [ ]:
import re
import time
from spotipy.oauth2 import SpotifyClientCredentials
from tqdm.notebook import tqdm
import spotipy
from fuzzywuzzy import fuzz

In [ ]:
client_credentials_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID, client_secret=CLIENT_SECRET
)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# Quick sanity check that auth actually works before running the full pipeline
try:
    _test = sp.search(q="test", limit=1, type="track")
    print("Spotify authentication OK.")
except Exception as e:
    print("Spotify authentication FAILED:", e)
    raise

## Load Song List

Input comes from `data/songs.csv`, produced by `2_combining_billboard_19000_datasets.ipynb`.

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, 'songs.csv'), index_col=0)
data.Title = data['Title'].str.lower()
data.Artist = data['Artist'].str.lower()
data.head()

# Get Spotify URI

For each song, search Spotify and fuzzy-match the result to confirm it's actually the same song (titles/artists from Wikipedia/Kaggle don't always match Spotify's naming exactly).

In [ ]:
titles = list(data.Title)
artists = list(data.Artist)

if SAMPLE_SIZE is not None:
    titles = titles[:SAMPLE_SIZE]
    artists = artists[:SAMPLE_SIZE]

print(f"Fetching URIs for {len(titles)} songs" + (" (SAMPLE_SIZE limited)" if SAMPLE_SIZE else " (full dataset)"))

In [ ]:
spotify_uri = list()
errors = list()

In [ ]:
def get_spotify_uri(title, artist):
    title_clean = re.sub(r"[,.;@#?!&$%()]+", ' ', title)
    title_clean = re.sub('\s+', ' ', title_clean).strip()
    artist_clean = re.sub('\s+', ' ', artist).strip()

    query = title_clean + " " + artist_clean

    try:
        search = sp.search(q=query, limit=50, offset=0, type='track', market='US')
    except Exception as e:
        print(f"Search failed for '{query}': {e}")
        return 0

    search_items = search['tracks']['items']

    for i in range(len(search_items)):
        spotify_title = search_items[i]['name']
        spotify_artist = search_items[i]['artists'][0]['name']

        spotify_title_clean = re.sub(r"[,.;@#?!&$%()]+", ' ', spotify_title)
        spotify_title_clean = re.sub('\s+', ' ', title_clean).strip().lower()
        spotify_artist_clean = spotify_artist.lower().strip().lower()

        fuzzy_title_match = fuzz.token_set_ratio(title_clean, spotify_title_clean)
        fuzzy_artist_match = fuzz.token_set_ratio(artist_clean, spotify_artist_clean)
        fuzzy_match = (fuzzy_title_match + fuzzy_artist_match) / 2

        if (fuzzy_title_match >= 90) and (fuzzy_artist_match >= 50) and fuzzy_match >= 75:
            uri = search_items[i]['id']
            return uri
    return 0

## Run the URI Search

Checkpoints progress to `data/songs_w_uri_checkpoint.csv` every 500 songs, so a crash partway through a long run doesn't lose everything.

In [ ]:
CHECKPOINT_EVERY = 500
checkpoint_path = os.path.join(DATA_DIR, 'songs_w_uri_checkpoint.csv')

temp = list()

for i in tqdm(range(len(titles))):
    uri = get_spotify_uri(titles[i], artists[i])

    if uri != 0:
        temp.append(uri)
    else:
        temp.append(uri)
        errors.append(i)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = data.iloc[:len(temp)].copy()
        checkpoint_df['URI'] = temp
        checkpoint_df.to_csv(checkpoint_path)
        print(f"Checkpoint saved at row {i + 1} ({len(errors)} errors so far)")

spotify_uri = spotify_uri + temp

print(f"Done. {len(errors)} / {len(titles)} songs had no confident match.")

In [ ]:
data = data.iloc[:len(spotify_uri)].copy()
data['URI'] = spotify_uri
data = data[data.URI != 0]

print(f"{len(data)} songs matched with a Spotify URI.")

In [ ]:
output_path = os.path.join(DATA_DIR, 'songs_w_uri.csv')
data.to_csv(output_path)
print(f"Saved {len(data)} rows to {os.path.abspath(output_path)}")

# Get Spotify Audio Features

**Changed from the original**: Spotify [restricted the Audio Features endpoint](https://developer.spotify.com/blog/2024-11-27-changes-to-the-web-api) for any app created on or after November 27, 2024 — a live `sp.audio_features(uri)` call will fail with a 403 for any newly-created app, permanently (not a quota/waiting issue).

This version tries the live API first (in case you ever get Extended Quota Mode access, or Spotify changes the policy again), and **falls back to `data/19000-spotify-songs/song_data.csv`** — the Kaggle file that already has these exact same 13 audio features for the same batch of ~18,835 songs — when the API call fails.

**Coverage caveat**: `song_data.csv` only covers the original 19,000-song dataset. Billboard-only songs that were added fresh during the scrape/merge steps won't be in it, so those rows will still end up with missing features and get dropped by `dropna()` below — same as they always would if a lookup simply failed. A `Feature_Source` column is added so you can see afterward how many songs came from the live API vs. the CSV fallback vs. neither.

In [ ]:
import os
print(os.listdir('data'))

In [27]:
data = pd.read_csv(os.path.join(DATA_DIR, 'songs_w_uri.csv'), index_col=0)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\songs_w_uri.csv'

## Load the CSV Fallback Table

In [ ]:
song_data_path = os.path.join(DATA_DIR, '19000-spotify-songs', 'song_data.csv')
song_data = pd.read_csv(song_data_path)

song_data['song_name_lower'] = song_data['song_name'].str.lower().str.strip()
song_data = song_data.drop_duplicates(subset='song_name_lower', keep='first')
song_data_lookup = song_data.set_index('song_name_lower').to_dict('index')

print(f"Loaded {len(song_data_lookup)} songs available as a fallback from {song_data_path}")

In [ ]:
uris = list(data.URI)
titles_for_features = list(data.Title)

danceability_list = list()
energy_list = list()
key_list = list()
loudness_list = list()
mode_list = list()
speechiness_list = list()
acousticness_list = list()
instrumentalness_list = list()
liveness_list = list()
valence_list = list()
tempo_list = list()
duration_list = list()
time_signature_list = list()
feature_source_list = list()

In [ ]:
# Maps this project's column names to song_data.csv's column names
CSV_FEATURE_MAP = {
    'danceability': 'danceability',
    'energy': 'energy',
    'key': 'key',
    'loudness': 'loudness',
    'mode': 'audio_mode',
    'speechiness': 'speechiness',
    'acousticness': 'acousticness',
    'instrumentalness': 'instrumentalness',
    'liveness': 'liveness',
    'valence': 'audio_valence',
    'tempo': 'tempo',
    'duration_ms': 'song_duration_ms',
    'time_signature': 'time_signature',
}

def get_audio_features_from_api(uri):
    try:
        search = sp.audio_features(uri)
    except Exception:
        return None

    if not search or search[0] is None:
        return None

    return search[0]

def get_audio_features_from_csv(song_title):
    row = song_data_lookup.get(song_title.lower().strip())
    if row is None:
        return None
    return {dest: row[src] for dest, src in CSV_FEATURE_MAP.items()}

def get_audio_features(uri, song_title):
    features = get_audio_features_from_api(uri)
    source = 'api'

    if features is None:
        features = get_audio_features_from_csv(song_title)
        source = 'csv' if features is not None else 'missing'

    if features is None:
        danceability_list.append(np.nan)
        energy_list.append(np.nan)
        key_list.append(np.nan)
        loudness_list.append(np.nan)
        mode_list.append(np.nan)
        speechiness_list.append(np.nan)
        acousticness_list.append(np.nan)
        instrumentalness_list.append(np.nan)
        liveness_list.append(np.nan)
        valence_list.append(np.nan)
        tempo_list.append(np.nan)
        duration_list.append(np.nan)
        time_signature_list.append(np.nan)
        feature_source_list.append(source)
        return

    danceability_list.append(features['danceability'])
    energy_list.append(features['energy'])
    key_list.append(features['key'])
    loudness_list.append(features['loudness'])
    mode_list.append(features['mode'])
    speechiness_list.append(features['speechiness'])
    acousticness_list.append(features['acousticness'])
    instrumentalness_list.append(features['instrumentalness'])
    liveness_list.append(features['liveness'])
    valence_list.append(features['valence'])
    tempo_list.append(features['tempo'])
    duration_list.append(features['duration_ms'])
    time_signature_list.append(features['time_signature'])
    feature_source_list.append(source)

In [ ]:
for i in tqdm(range(len(uris))):
    get_audio_features(uris[i], titles_for_features[i])

print(pd.Series(feature_source_list).value_counts())

In [ ]:
data['Danceability'] = danceability_list
data['Energy'] = energy_list
data['Key'] = key_list
data['Loudness'] = loudness_list
data['Mode'] = mode_list
data['Speechiness'] = speechiness_list
data['Acousticness'] = acousticness_list
data['Instrumentalness'] = instrumentalness_list
data['Liveness'] = liveness_list
data['Valence'] = valence_list
data['Tempo'] = tempo_list
data['Duration'] = duration_list
data['Time_Signature'] = time_signature_list
data['Feature_Source'] = feature_source_list

data = data.dropna(subset=[
    'Danceability', 'Energy', 'Key', 'Loudness', 'Mode', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo',
    'Duration', 'Time_Signature'
])
print(f"{len(data)} songs with complete audio features.")

In [ ]:
output_path = os.path.join(DATA_DIR, 'songs_w_spotifyapi.csv')
data.to_csv(output_path)
print(f"Saved {len(data)} rows to {os.path.abspath(output_path)}")

# Get Release Year

**Changed from the original**: rather than re-searching Spotify by title/artist a second time (which risks landing on a *different* track than the one already matched, and costs an extra API call per song), this looks up the release year directly from the URI already confirmed in the "Get Spotify URI" step via `sp.track(uri)`.

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, 'songs_w_spotifyapi.csv'), index_col=0)

In [ ]:
def get_song_year(uri):
    try:
        track = sp.track(uri)
        return track['album']['release_date']
    except Exception as e:
        print(f"Track lookup failed for {uri}: {e}")
        return 0

def clean_year(date):
    y = date.split('-')[0]
    return int(y)

In [ ]:
uris = list(data.URI)
years = list()
errors = list()

for i in tqdm(range(len(uris))):
    year = get_song_year(uris[i])

    if year != 0:
        years.append(year)
    else:
        years.append(year)
        print("Errored on " + str(i))
        errors.append(i)

In [ ]:
new_years = list()

for y in years:
    if y == 0:
        new_years.append(y)
    else:
        new_years.append(clean_year(y))

data['Release_Year'] = new_years
data = data[data.Release_Year != 0]

print(f"{len(data)} songs with a release year.")

In [ ]:
output_path = os.path.join(DATA_DIR, 'songs_w_features_year.csv')
data.to_csv(output_path)
print(f"Saved {len(data)} rows to {os.path.abspath(output_path)}")

# Get Genre

Approximates each song's genre by looking up its (first-listed) artist's Spotify genre tags and mapping them to a fixed list of common genre keywords.

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, 'songs_w_features_year.csv'), index_col=0)

In [ ]:
artists = list(data.Artist)

In [ ]:
def get_first_artist(artist):

    # Handling , or and
    for i in range(len(artist)):
        if artist[i] == ',' or artist[i] == '&':
              return artist[0:i]
        if artist[i:i+3] == 'and':
              return artist[0:i]
    return artist

# Helper function to take subgenres of each artist and find the most frequent common genre within potentially 1000 subgenres
def get_common_genres(test_subgenres, common_genre_keywords):

    final_genres = []
    genre_frequency_map = {}

    # Checking the subgenres to see if they match common genre keywords
    for keyword in common_genre_keywords:
        for subg in test_subgenres:
            if keyword in subg:
                final_genres.append(keyword)

    # If no final genres can be identified, return None.
    if len(final_genres) == 0:
        return "None"

    # Counting the number of each genre keyword and returning the one with the highest count (as the "common genre")
    for genre in final_genres:
        if genre in genre_frequency_map.keys():
            current_value = genre_frequency_map[genre]
            genre_frequency_map[genre] = current_value + 1
        else:
            genre_frequency_map[genre] = 1

    # Getting most frequent common genre.
    max_value = max(genre_frequency_map.values())  # maximum value
    max_keys = [k for k, v in genre_frequency_map.items() if v == max_value]
    return max_keys[0]

In [ ]:
def get_genres_for_each_artist(artists):

    final_genre_list = []
    # Most popular American Music Genres
    common_genre_keywords = ["rap", "pop", "rock", "country", "alternative", "r&b" "latin", "edm", "seasonal", "jazz", "classical", "metal", "reggae"]

    for i in tqdm(range(len(artists))):
        artist_name = artists[i]
        # Clean Artist Name
        artist_name = get_first_artist(artist_name)

        # Using the spotify search functionality and extracting the list of subgenres.
        try:
            search = sp.search(q=artist_name, limit=1, offset=0, type='artist', market='US')
        except Exception:
            print('Spotify error for ' + str(i))
            final_genre_list.append('None')
            continue

        try:
            list_of_subgenres = search['artists']['items'][0]['genres']
            common_genre = get_common_genres(list_of_subgenres, common_genre_keywords)
            final_genre_list.append(common_genre)
        except IndexError:
            print('No Genre for ' + str(i))
            final_genre_list.append('None')
            continue

    return final_genre_list

In [ ]:
genres_list = get_genres_for_each_artist(artists)

In [ ]:
data['Genre'] = genres_list
data = data[data['Genre'] != 'None']

print(f"{len(data)} songs with a genre assigned.")

## Save Final Output

This is `songs_complete_data.csv` — the dataset used by `eda_4.ipynb` and `ML_top100.ipynb`.

In [ ]:
output_path = os.path.join(DATA_DIR, 'songs_complete_data.csv')
data.to_csv(output_path)
print(f"Saved {len(data)} rows to {os.path.abspath(output_path)}")

## Next Steps

- If you ran this with `SAMPLE_SIZE` set to a small number, set it to `None` at the top of the notebook and re-run the whole thing top to bottom to build the full dataset.
- The full run can take a while (roughly 3 API-call stages × up to ~14,000 songs each), so consider running it in a terminal via `jupyter nbconvert --to notebook --execute` if you don't want to keep the browser/VS Code window open, or just let it run in the background — the checkpointing in the URI step means a crash won't cost you everything.